# 01: Explore

Messy exploration lives here: try things, look at the data, take notes. Anything that works and that I'd want again moves into `src/ridership/` and is imported back.

**My question:** How do temperature, rain and weekends affect daily rides, and can we predict ridership for unseen days?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ridership.load import load_rides

sns.set_theme(style="whitegrid", palette="colorblind")
raw = load_rides("../data/raw/daily_rides.csv")
print(raw.shape)
raw.head()

In [ ]:
raw.info()
raw.describe().round(1)

## Things that look odd (my cleaning to-do list)
* `temp_c` has missing values. How many, and are they clustered in time?
* Is `is_weekend` consistent with the calendar?
* Are there impossible values (negative rain, zero rides)?

In [ ]:
missing = raw[raw["temp_c"].isna()]
print(len(missing), "missing temperatures on:", [d.strftime("%b %d") for d in missing["date"]])
print("weekend flag consistent:", ((raw["date"].dt.dayofweek >= 5).astype(int) == raw["is_weekend"]).all())
print("negative rain:", (raw["rain_mm"] < 0).sum(), "| non-positive rides:", (raw["rides"] <= 0).sum())

Six scattered missing days, nothing impossible. Plan: flag them and fill with the month's median (`clean.py`), and later check the results don't depend on that choice.

## First looks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
sns.histplot(raw["rides"], bins=25, ax=axes[0]); axes[0].set_title("Distribution of daily rides")
sns.scatterplot(data=raw, x="temp_c", y="rides", hue="is_weekend", alpha=0.6, ax=axes[1]); axes[1].set_title("Rides vs temperature")
sns.scatterplot(data=raw, x="rain_mm", y="rides", alpha=0.5, ax=axes[2]); axes[2].set_title("Rides vs rain: no obvious pattern")
plt.tight_layout(); plt.show()
print(raw[["temp_c", "rain_mm", "is_weekend", "rides"]].corr().round(2)["rides"])

**Observations.** Rides are roughly bell-shaped; temperature has a very strong relationship with rides; weekends sit lower; rain shows almost nothing on its own (r ≈ −0.02).
**New question:** is rain really irrelevant, or is its effect hidden by temperature? (This became the regression in `analysis.py`.)